# GitHub PR Tracker - Backend Technical Requirements

**Project:** GitHub PR Tracker Dashboard  
**Document:** Backend Technical Requirements  
**Version:** 3.0 (Phase 1 + Phase 2 + Phase 3)  
**Date:** 15 June 2026  
**Tech Stack:** Python + FastAPI

## 1. Tech Stack Summary

| Component | Technology | Version | Purpose |
|-----------|-----------|---------|--------|
| **Language** | Python | 3.11+ | Core runtime |
| **Framework** | FastAPI | 0.100+ | REST API framework (async, auto-docs) |
| **HTTP Client** | httpx | 0.27+ | Async HTTP client for GitHub API calls |
| **GitHub SDK** | PyGithub or httpx (direct) | — | GitHub API integration |
| **Server** | Uvicorn | 0.30+ | ASGI server |
| **Validation** | Pydantic v2 | 2.0+ | Request/response models |
| **Environment** | python-dotenv | — | Configuration management |
| **Testing** | pytest + pytest-asyncio | — | Unit and integration tests |
| **Linting** | Ruff | — | Fast linter + formatter |

> **Note (Phase 1):** No in-memory caching layer. All requests fetch fresh data from GitHub API.  
> With ~10 team members and 5,000 req/hr rate limit, caching is unnecessary for Phase 1.  
> Can be revisited in Phase 2 if rate limits become a concern.

## 2. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                     Frontend (React + Vite)                   │
└─────────────────────────┬───────────────────────────────────┘
                          │ REST API (JSON)
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   FastAPI Backend                             │
│                                                              │
│  ┌──────────┐  ┌──────────────┐  ┌───────────────────────┐ │
│  │  Routes  │  │  Services    │  │  GitHub Client        │ │
│  │  (API)   │──│  (Business   │──│  (httpx + GitHub API) │ │
│  │          │  │   Logic)     │  │                       │ │
│  └──────────┘  └──────────────┘  └───────────────────────┘ │
│                                                              │
│  ┌──────────────────────────────────────────────────────────┐│
│  │  CODEOWNERS Parser (Glob matching)                        ││
│  └──────────────────────────────────────────────────────────┘│
└─────────────────────────┬───────────────────────────────────┘
                          │ HTTPS
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   GitHub REST API (api.github.com)            │
└─────────────────────────────────────────────────────────────┘
```

**Key:** No caching layer in Phase 1. Backend always fetches fresh data from GitHub on each request.

## 3. Project Structure

```
backend/
├── app/
│   ├── __init__.py
│   ├── main.py                 # FastAPI app entry point
│   ├── config.py               # Configuration (env vars, settings)
│   ├── routes/
│   │   ├── __init__.py
│   │   ├── pull_requests.py    # PR endpoints
│   │   ├── team.py             # Team member endpoints
│   │   └── health.py           # Health check
│   ├── services/
│   │   ├── __init__.py
│   │   ├── github_client.py    # GitHub API wrapper (httpx)
│   │   ├── pr_service.py       # PR business logic (fetch, filter, enrich)
│   │   ├── team_service.py     # Team member resolution
│   │   └── codeowners.py       # CODEOWNERS parser + matcher
│   ├── models/
│   │   ├── __init__.py
│   │   ├── pr.py               # PR Pydantic models
│   │   ├── team.py             # Team member models
│   │   └── config.py           # Config models
│   └── utils/
│       ├── __init__.py
│       ├── time_helpers.py     # Age calculation utilities
│       └── glob_matcher.py     # CODEOWNERS glob pattern matching
├── tests/
│   ├── __init__.py
│   ├── test_pr_service.py
│   ├── test_codeowners.py
│   ├── test_team_service.py
│   └── test_routes.py
├── .env.example                # Environment variable template
├── requirements.txt            # Python dependencies
├── pyproject.toml              # Project metadata + tool config
└── Dockerfile                  # Container build (Linux target)
```

## 4. API Endpoints

### 4.1 Pull Requests

| Method | Endpoint | Description | Query Params |
|--------|----------|-------------|-------------|
| `GET` | `/api/v1/pull-requests` | Fetch all open PRs for the team | `branch_type` (all/main/feature), `sort_by` (age/author/reviewers), `sort_order` (asc/desc) — all optional, defaults to all/age/desc |
| `GET` | `/api/v1/pull-requests/{pr_number}` | Fetch detailed info for a single PR | — |

> **Phase 2 Note:** The backend **retains** the `branch_type`, `sort_by`, `sort_order` query parameters for backward compatibility and API consumers. The frontend simply stops sending them (fetches all PRs and filters client-side). No backend API contract change.

### 4.2 Team

| Method | Endpoint | Description | Query Params |
|--------|----------|-------------|-------------|
| `GET` | `/api/v1/team/members` | Get list of resolved team members | — |

### 4.3 Health

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/api/v1/health` | Health check (includes GitHub API rate limit status) |

## 5. API Response Models

### 5.1 GET `/api/v1/pull-requests` Response

```json
{
  "total_count": 12,
  "pull_requests": [
    {
      "number": 142,
      "title": "Add user authentication module",
      "author": {
        "username": "johndoe",
        "avatar_url": "https://avatars.githubusercontent.com/u/12345"
      },
      "base_branch": "main",
      "head_branch": "feature/auth-module",
      "branch_type": "main",
      "created_at": "2026-05-29T10:30:00Z",
      "age": {
        "days": 4,
        "hours": 2,
        "display": "4d 2h",
        "is_stale": true,
        "threshold_days": 3
      },
      "active_reviewers_count": 2,
      "team_approvals": {
        "count": 2,
        "required": 2,
        "sufficient": true,
        "reviewers": ["user1", "user2"]
      },
      "unresolved_comment_count": 1,
      "code_owner_status": {
        "required": true,
        "approved": false
      },
      "html_url": "https://github.com/org/repo/pull/142",
      "labels": ["enhancement", "backend"]
    }
  ]
}
```

> **Phase 2 Changes:**
> - Removed `filters_applied` field (filtering is now client-side).
> - Added `team_approvals` object with `count`, `required`, `sufficient`, and `reviewers` fields.
> - Added `unresolved_comment_count` integer field.

### 5.2 GET `/api/v1/pull-requests/{pr_number}` Response (Detail)

```json
{
  "number": 142,
  "title": "Add user authentication module",
  "body": "## Summary\nThis PR adds the authentication module...",
  "author": {
    "username": "johndoe",
    "avatar_url": "https://avatars.githubusercontent.com/u/12345"
  },
  "base_branch": "main",
  "head_branch": "feature/auth-module",
  "branch_type": "main",
  "created_at": "2026-05-29T10:30:00Z",
  "age": {
    "days": 4,
    "hours": 2,
    "display": "4d 2h",
    "is_stale": true,
    "threshold_days": 3
  },
  "active_reviewers": [
    {
      "username": "reviewer1",
      "avatar_url": "https://avatars.githubusercontent.com/u/111",
      "state": "APPROVED"
    },
    {
      "username": "reviewer2",
      "avatar_url": "https://avatars.githubusercontent.com/u/222",
      "state": "COMMENTED"
    }
  ],
  "active_reviewers_count": 2,
  "team_approvals": {
    "count": 2,
    "required": 2,
    "sufficient": true,
    "reviewers": ["reviewer1", "alice"]
  },
  "unresolved_comment_count": 3,
  "unresolved_threads": [
    {
      "commenter": "alice",
      "file_path": "src/auth/login.ts",
      "comment_snippet": "Should we add rate limiting here?",
      "url": "https://github.com/org/repo/pull/142#discussion_r123"
    },
    {
      "commenter": "bob",
      "file_path": "src/api/client.ts",
      "comment_snippet": "This error handling doesn't cover 403...",
      "url": "https://github.com/org/repo/pull/142#discussion_r124"
    },
    {
      "commenter": "carol",
      "file_path": null,
      "comment_snippet": "Can you add tests for the edge case?",
      "url": "https://github.com/org/repo/pull/142#discussion_r125"
    }
  ],
  "code_owner_status": {
    "required": true,
    "approved": false,
    "owners": [
      { "username": "codeowner1", "has_approved": false },
      { "username": "codeowner2", "has_approved": false }
    ]
  },
  "files_changed": {
    "total": 8,
    "additions": 245,
    "deletions": 32
  },
  "labels": ["enhancement", "backend"],
  "html_url": "https://github.com/org/repo/pull/142"
}
```

> **Phase 2 additions:** `team_approvals`, `unresolved_comment_count`, and `unresolved_threads` fields added to the detail response.

## 6. GitHub API Integration

### 6.1 APIs Used

| # | GitHub API Endpoint | Purpose | Phase |
|---|--------------------|---------|-------|
| 1 | `GET /repos/{owner}/{repo}/pulls?state=open` | Fetch open PRs (handles pagination server-side) | 1 |
| 2 | `GET /orgs/{org}/teams/{team_slug}/members` | Fetch team members | 1 |
| 3 | `GET /repos/{owner}/{repo}/pulls/{number}/reviews` | Fetch PR reviews | 1 |
| 4 | `GET /repos/{owner}/{repo}/pulls/{number}/comments` | Fetch PR comments | 1 |
| 5 | `GET /repos/{owner}/{repo}/pulls/{number}/files` | Fetch changed files | 1 |
| 6 | `GET /repos/{owner}/{repo}/contents/.github/CODEOWNERS` | Fetch CODEOWNERS file | 1 |
| 7 | `GET /rate_limit` | Check API rate limit | 1 |
| 8 | `POST /graphql` — `repository.pullRequest.reviewThreads` | Fetch unresolved review thread count | **2** |

### 6.2 Rate Limiting (Phase 1 Decision)

- GitHub API allows **5,000 requests/hour** for authenticated requests.
- With ~10 team members and a small number of PRs, we will **never approach** this limit.
- **No caching / TTL in Phase 1** — all data fetched fresh on each API call.
- Rate limit info is still exposed via the `/api/v1/health` endpoint for observability.
- **Phase 2 note:** GraphQL API has a separate cost-based rate limit (5,000 points/hour). Each `reviewThreads` query costs ~1 point. Well within limits.

### 6.3 Pagination (Server-side)

- GitHub returns max 30 PRs per page by default (configurable up to 100).
- Backend handles pagination internally using `per_page=100` and `Link` header / page iteration.
- Frontend receives a single consolidated response (no client-side pagination needed in Phase 1).

### 6.4 Authentication

- GitHub PAT (Personal Access Token) passed via `Authorization: Bearer {token}` header.
- Token stored in environment variable `GITHUB_TOKEN`.
- Required scopes: `repo`, `read:org`.
- Same token works for both REST API and GraphQL API (`POST https://api.github.com/graphql`).

## 7. Core Services Logic

### 7.1 PR Service (`pr_service.py`)

```python
# Pseudocode flow for fetching dashboard data

async def get_team_pull_requests(branch_type, sort_by, sort_order):
    # 1. Get team members
    team_members = await team_service.get_members()
    
    # 2. Fetch all open PRs
    all_prs = await github_client.get_open_prs()
    
    # 3. Filter by team members
    team_prs = [pr for pr in all_prs if pr.author in team_members]
    
    # 4. Filter by branch type
    if branch_type != "all":
        team_prs = filter_by_branch_type(team_prs, branch_type)
    
    # 5. Enrich each PR with:
    for pr in team_prs:
        pr.age = calculate_age(pr.created_at)
        pr.is_stale = check_staleness(pr.age, pr.branch_type)
        pr.active_reviewers_count = await get_active_reviewer_count(pr)
        pr.code_owner_status = await check_code_owner_approval(pr)
    
    # 6. Sort
    team_prs = sort_prs(team_prs, sort_by, sort_order)
    
    return team_prs
```

### 7.2 Active Reviewer Logic

```python
async def get_active_reviewer_count(pr):
    reviews = await github_client.get_pr_reviews(pr.number)
    comments = await github_client.get_pr_comments(pr.number)
    
    # Unique users who have reviewed (APPROVED, CHANGES_REQUESTED)
    # or commented (excluding the PR author)
    active_users = set()
    
    for review in reviews:
        if review.state in ("APPROVED", "CHANGES_REQUESTED"):
            active_users.add(review.user)
    
    for comment in comments:
        if comment.user != pr.author:
            active_users.add(comment.user)
    
    return len(active_users)
```

### 7.3 Staleness Check

```python
def check_staleness(age_days: float, branch_type: str) -> bool:
    thresholds = {
        "main": config.AGING_THRESHOLD_MAIN,      # default: 3 days
        "feature": config.AGING_THRESHOLD_FEATURE  # default: 2 days
    }
    return age_days >= thresholds.get(branch_type, 3)
```

## 8. CODEOWNERS Parser

### 8.1 Parsing Logic

```python
# CODEOWNERS file format:
# <pattern> <owner1> <owner2> ...
# Examples:
#   *           @org/global-owners
#   *.js        @frontend-team
#   /src/api/   @backend-team @user1

def parse_codeowners(content: str) -> list[CodeOwnerRule]:
    rules = []
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        pattern = parts[0]
        owners = parts[1:]  # @user or @org/team
        rules.append(CodeOwnerRule(pattern=pattern, owners=owners))
    return rules
```

### 8.2 File Matching

- Use `fnmatch` or `pathlib.PurePath.match()` for glob pattern matching.
- Last matching rule wins (same as GitHub's behavior).
- Support patterns: `*`, `**`, `/path/`, `*.ext`.

### 8.3 Fetch Strategy

- CODEOWNERS fetched fresh on each PR detail request (no caching in Phase 1).
- The file rarely changes, so the API overhead is minimal (1 extra call per detail view).
- Team member resolution for `@org/team` owners uses the same team API call.

## 9. Configuration

### Environment Variables (`.env`)

```env
# GitHub
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
GITHUB_ORG=my-org
GITHUB_REPO=my-repo
GITHUB_TEAM_SLUG=phoenix

# Aging Thresholds (days)
AGING_THRESHOLD_MAIN=3
AGING_THRESHOLD_FEATURE=2

# Team Approvals (Phase 2)
REQUIRED_TEAM_APPROVALS=2

# Server
HOST=0.0.0.0
PORT=8000
CORS_ORIGINS=http://localhost:5173

# Optional: Fallback team members (comma-separated)
FALLBACK_TEAM_MEMBERS=user1,user2,user3

# Optional: Excluded team members (not on the project team)
EXCLUDED_TEAM_MEMBERS=
```

> **Phase 2 addition:** `REQUIRED_TEAM_APPROVALS` — configurable threshold for team approval sufficiency indicator.

## 10. Error Handling

| Scenario | Handling |
|----------|----------|
| GitHub API rate limit exceeded | Return 429 with message; log warning (unlikely in Phase 1) |
| GitHub token invalid/expired | Return 401 with clear error message |
| Team API fails (permissions) | Fall back to `FALLBACK_TEAM_MEMBERS` config |
| CODEOWNERS file not found | Skip code owner status; return `code_owner_status: null` |
| Network timeout to GitHub | Retry once with exponential backoff; then return error |
| Invalid PR number | Return 404 |

## 11. CORS Configuration

Since frontend (Vite on port 5173) and backend (FastAPI on port 8000) run on different ports during development:

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],  # Vite dev server
    allow_credentials=True,
    allow_methods=["GET"],
    allow_headers=["*"],
)
```

## 12. Performance Considerations

| Concern | Solution |
|---------|----------|
| Multiple GitHub API calls per PR (reviews, comments, files) | Use `asyncio.gather()` for parallel requests |
| Large number of PRs | Paginate GitHub API calls internally (`per_page=100`); process in batches |
| CODEOWNERS parsing for every PR | Parse CODEOWNERS once per request cycle; reuse parsed result across PR evaluations |
| Cold start | No pre-warming needed (no cache layer) |

> **Phase 2 consideration:** If latency becomes noticeable, add TTL-based in-memory caching for team members (changes rarely) and CODEOWNERS file.

## 13. Testing Strategy

| Test Type | Tools | Coverage |
|-----------|-------|----------|
| **Unit tests** | pytest | Services, CODEOWNERS parser, age calculation, glob matcher |
| **Integration tests** | pytest + httpx (TestClient) | API endpoints with mocked GitHub responses |
| **Mocking** | pytest-mock / respx | Mock GitHub API responses |

Key test cases:
- CODEOWNERS parser handles all pattern types
- Active reviewer count excludes non-active reviewers
- Branch classification is correct
- Staleness thresholds apply correctly per branch type
- Fallback works when team API fails

## 14. Deployment

### Target: Linux Machine (Team-wide access)

The application will be deployed on a Linux server to be accessible by all team members.

### Dockerfile

```dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app/

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Running Locally (Development)

```bash
# Install dependencies
pip install -r requirements.txt

# Run server
uvicorn app.main:app --reload --port 8000

# API docs available at:
# http://localhost:8000/docs (Swagger)
# http://localhost:8000/redoc (ReDoc)
```

### Production Deployment (Linux)

```bash
# Build Docker image
docker build -t pr-tracker-backend .

# Run container
docker run -d \
  --name pr-tracker-backend \
  -p 8000:8000 \
  --env-file .env \
  --restart unless-stopped \
  pr-tracker-backend

# Or run directly with systemd (alternative)
# Create /etc/systemd/system/pr-tracker-backend.service
```

### docker-compose.yml (Full Stack)

```yaml
version: "3.8"
services:
  backend:
    build: ./backend
    ports:
      - "8000:8000"
    env_file: ./backend/.env
    restart: unless-stopped
    
  frontend:
    build: ./frontend
    ports:
      - "80:80"
    depends_on:
      - backend
    restart: unless-stopped
```

## 15. Dependencies (`requirements.txt`)

```
fastapi>=0.100.0
uvicorn[standard]>=0.30.0
httpx>=0.27.0
pydantic>=2.0.0
pydantic-settings>=2.0.0
python-dotenv>=1.0.0

# Dev dependencies
pytest>=7.0.0
pytest-asyncio>=0.21.0
respx>=0.20.0
ruff>=0.1.0
```

> **Removed from Phase 1:** `cachetools` (no caching layer needed)

## 16. Phase 2 Backend Changes

### 16.1 Backend Filter Params — Retained (No Breaking Change)

The backend **keeps** the `branch_type`, `sort_by`, `sort_order` query parameters unchanged. The API contract is not modified. The frontend simply stops sending these params and performs filtering/sorting client-side.

```python
# Unchanged — backend still supports this:
@router.get("", response_model=PRListResponse)
async def list_pull_requests(
    branch_type: str = Query(default="all", pattern="^(all|main|feature)$"),
    sort_by: str = Query(default="age", pattern="^(age|author|reviewers)$"),
    sort_order: str = Query(default="desc", pattern="^(asc|desc)$"),
) -> PRListResponse:
```

When the frontend calls without params, the defaults (`all`, `age`, `desc`) return all PRs sorted by age — which is exactly what the frontend needs for client-side filtering.

### 16.2 Team Approvals Computation

New logic in `pr_service.py`. Team membership is determined by the PHOENIX GitHub team (same team used to filter which PRs appear on the dashboard).

```python
async def compute_team_approvals(pr, team_members, code_owners):
    """
    Count APPROVED reviews from PHOENIX team members, excluding:
    - The PR author (even if they're a team member)
    - Code owners (they have their own status column)
    
    team_members: fetched from GET /orgs/{org}/teams/{team_slug}/members (PHOENIX)
    """
    reviews = await github_client.get_pr_reviews(pr.number)
    
    approved_by = []
    for review in reviews:
        if (review.state == "APPROVED"
            and review.user in team_members
            and review.user != pr.author
            and review.user not in code_owners):
            approved_by.append(review.user)
    
    # Deduplicate (user may approve multiple times)
    approved_by = list(dict.fromkeys(approved_by))
    
    required = config.REQUIRED_TEAM_APPROVALS  # default: 2
    return {
        "count": len(approved_by),
        "required": required,
        "sufficient": len(approved_by) >= required,
        "reviewers": approved_by  # Names shown in tooltip on frontend
    }
```

> **Key:** "Team" = PHOENIX GitHub team. The same membership list already used in Phase 1 to filter PRs by author.

### 16.3 Unresolved Comments (GitHub GraphQL API)

Uses the GitHub GraphQL API for accurate unresolved thread counts.

**For PR list (count only):**
```graphql
query($owner: String!, $repo: String!, $number: Int!) {
  repository(owner: $owner, name: $repo) {
    pullRequest(number: $number) {
      reviewThreads(first: 100) {
        nodes {
          isResolved
        }
      }
    }
  }
}
```

**For PR detail (full thread info for the detail panel):**
```graphql
query($owner: String!, $repo: String!, $number: Int!) {
  repository(owner: $owner, name: $repo) {
    pullRequest(number: $number) {
      reviewThreads(first: 100) {
        nodes {
          isResolved
          comments(first: 1) {
            nodes {
              author { login }
              body
              path
              url
            }
          }
        }
      }
    }
  }
}
```

```python
async def get_unresolved_comment_count(pr_number: int) -> int:
    """Count unresolved review threads using GitHub GraphQL API."""
    # ... returns integer count for list view

async def get_unresolved_threads(pr_number: int) -> list[UnresolvedThread]:
    """Get full unresolved thread details for detail panel."""
    # Returns: commenter, file path, comment snippet, GitHub URL
```

> **Note:** The same GitHub PAT works for both REST and GraphQL APIs. GraphQL endpoint: `https://api.github.com/graphql`. Requires `repo` scope (already required).

### 16.4 New Environment Variable

| Variable | Default | Description |
|----------|---------|-------------|
| `REQUIRED_TEAM_APPROVALS` | `2` | Number of team approvals needed before code owner ping |

Added to `.env.example` and `config.py`:
```python
REQUIRED_TEAM_APPROVALS: int = 2
```

### 16.5 Updated Response Models

**PR List Response (adds new fields, keeps `filters_applied`):**

```python
class TeamApprovals(BaseModel):
    count: int
    required: int
    sufficient: bool
    reviewers: list[str]  # Names for tooltip hover

class PullRequestResponse(BaseModel):
    # ... existing fields ...
    team_approvals: TeamApprovals          # NEW
    unresolved_comment_count: int          # NEW
```

**PR Detail Response (adds Phase 2 sections):**

```python
class UnresolvedThread(BaseModel):
    commenter: str         # GitHub username who started the thread
    file_path: str | None  # File the comment is on (null for PR-level comments)
    comment_snippet: str   # First ~100 chars of the comment body
    url: str               # Direct link to the comment on GitHub

class PullRequestDetailResponse(BaseModel):
    # ... existing fields (body, active_reviewers, code_owner_status, files_changed) ...
    team_approvals: TeamApprovals               # NEW — full breakdown
    unresolved_threads: list[UnresolvedThread]   # NEW — full thread details
    unresolved_comment_count: int                # NEW — count (same as list view)
```

## 17. Phase 3 Backend Changes — Multi-Repository Support

### 17.1 Overview of Changes

Phase 3 replaces the single-repo `GET /repos/{owner}/{repo}/pulls` approach with the GitHub **Search API** to discover PRs across all repositories in the org where team members have open PRs.

| Area | Before (Phase 1-2) | After (Phase 3) |
|------|-------------------|-----------------|
| PR Discovery | `GET /repos/{org}/{repo}/pulls?state=open` | `GET /search/issues?q=is:pr+is:open+org:{org}+author:{user}` |
| Config | `GITHUB_REPO` required | `GITHUB_REPO` optional (filter mode) |
| Scope | Single repo | All org repos dynamically |
| Response | No repo field | `repository` field added to each PR |
| CODEOWNERS | Fetched from one repo | Fetched & cached per repo |

### 17.2 Configuration Changes

```env
# REMOVED (now optional):
# GITHUB_REPO=my-repo          ← No longer required

# UNCHANGED:
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
GITHUB_ORG=nasuni
GITHUB_TEAM_SLUG=phoenix
AGING_THRESHOLD_MAIN=3
AGING_THRESHOLD_FEATURE=2
REQUIRED_TEAM_APPROVALS=2
```

```python
# config.py changes
class Settings(BaseSettings):
    github_token: str
    github_org: str
    github_repo: Optional[str] = None  # ← Changed: now Optional, default None
    github_team_slug: str
    # ... rest unchanged
```

> **Backward compatibility:** If `GITHUB_REPO` is set, the system operates in single-repo mode (same as Phase 1-2). If unset/empty, it discovers all repos.

### 17.3 New GitHub Client Method — Search API

```python
async def search_team_pull_requests(self, team_usernames: list[str]) -> list[dict]:
    """
    Use GitHub Search API to find all open PRs by team members across the org.
    
    Strategy: Build query with OR'd authors:
      is:pr is:open org:{org} author:user1 author:user2 ...
    
    Note: GitHub Search API uses OR logic for multiple author: qualifiers.
    Rate limit: 30 requests/minute (authenticated).
    Max results: 1000 (sufficient for team open PRs).
    """
    # Build search query
    author_clauses = " ".join(f"author:{u}" for u in team_usernames)
    query = f"is:pr is:open org:{settings.github_org} {author_clauses}"
    
    results = []
    page = 1
    async with self._client() as client:
        while True:
            response = await client.get(
                "/search/issues",
                params={"q": query, "per_page": 100, "page": page},
            )
            response.raise_for_status()
            data = response.json()
            results.extend(data["items"])
            if len(results) >= data["total_count"]:
                break
            page += 1
    
    return results
```

> **Note:** The Search API returns issues/PRs in a slightly different format than the PRs API. Key differences:
> - `pull_request.html_url` exists to identify as PR
> - `repository_url` field contains the repo API URL (extract repo name from it)
> - No `base`/`head` branch info in search results — requires individual PR fetch for enrichment

### 17.4 PR Discovery Flow (Updated)

```python
async def get_team_pull_requests(...) -> PRListResponse:
    # 1. Get team members
    team_usernames = await get_team_usernames()
    
    if settings.github_repo:
        # LEGACY MODE: Single-repo (same as Phase 1-2)
        all_prs = await github_client.get_open_pull_requests()
        team_prs = [pr for pr in all_prs if pr["user"]["login"] in team_usernames]
    else:
        # PHASE 3: Multi-repo discovery via Search API
        search_results = await github_client.search_team_pull_requests(list(team_usernames))
        # Search results need enrichment — fetch full PR data per PR
        team_prs = await _fetch_full_pr_data(search_results)
    
    # 2. Filter drafts
    team_prs = [pr for pr in team_prs if not pr.get("draft", False)]
    
    # 3. Enrich (age, reviewers, code owners, team approvals, unresolved)
    enriched = await asyncio.gather(*[_enrich_pr_summary(pr, team_usernames) for pr in team_prs])
    
    # 4. Sort + return
    ...
```

### 17.5 Full PR Data Fetch (Search → REST)

The Search API doesn't return branch info (`base`, `head`). We need to fetch the full PR object:

```python
async def _fetch_full_pr_data(search_results: list[dict]) -> list[dict]:
    """
    For each search result, fetch the full PR data from REST API.
    The search result gives us the repo and PR number.
    """
    async def fetch_one(item: dict) -> dict:
        # Extract repo from repository_url: "https://api.github.com/repos/nasuni/portal"
        repo_name = item["repository_url"].split("/repos/")[-1]  # "nasuni/portal"
        owner, repo = repo_name.split("/")
        pr_number = item["number"]
        
        async with github_client._client() as client:
            response = await client.get(f"/repos/{owner}/{repo}/pulls/{pr_number}")
            response.raise_for_status()
            return response.json()
    
    return await asyncio.gather(*[fetch_one(item) for item in search_results])
```

### 17.6 Updated Response Models

**PR Summary (adds `repository` field):**

```python
class PullRequestSummary(BaseModel):
    number: int
    repository: str              # NEW — short repo name (e.g., "portal")
    title: str
    author: Author
    base_branch: str
    head_branch: str
    branch_type: str
    created_at: str
    age: AgeInfo
    active_reviewers_count: int
    code_owner_status: Optional[CodeOwnerStatus] = None
    team_approvals: Optional[TeamApprovals] = None
    unresolved_comment_count: int = 0
    html_url: str
    labels: list[str]
```

**PR Detail (adds `repository` field):**

```python
class PullRequestDetail(BaseModel):
    number: int
    repository: str              # NEW
    title: str
    # ... all existing fields ...
```

### 17.7 CODEOWNERS — Per-Repository Caching

```python
# In-memory cache for CODEOWNERS per repo (refreshed every cycle)
_codeowners_cache: dict[str, str | None] = {}

async def get_codeowners_for_repo(owner: str, repo: str) -> str | None:
    """Fetch CODEOWNERS for a specific repo, with per-cycle cache."""
    cache_key = f"{owner}/{repo}"
    if cache_key in _codeowners_cache:
        return _codeowners_cache[cache_key]
    
    content = await github_client.get_codeowners(owner, repo)
    _codeowners_cache[cache_key] = content
    return content
```

The `github_client.get_codeowners()` method is updated to accept `owner` and `repo` parameters instead of using `settings.github_repo`.

### 17.8 Updated GitHub Client Methods (Parameterized by Repo)

All per-PR methods now accept `owner`/`repo` parameters:

```python
async def get_pr_reviews(self, owner: str, repo: str, pr_number: int) -> list[dict]:
    ...

async def get_pr_comments(self, owner: str, repo: str, pr_number: int) -> list[dict]:
    ...

async def get_pr_files(self, owner: str, repo: str, pr_number: int) -> list[dict]:
    ...

async def get_codeowners(self, owner: str, repo: str) -> str | None:
    ...

async def get_review_threads(self, owner: str, repo: str, pr_number: int) -> list[dict]:
    ...
```

### 17.9 API Endpoint Changes

The `/api/v1/pull-requests/{pr_number}` detail endpoint needs to accept the repository context:

```python
# Option A: Add repo as query param
@router.get("/{pr_number}")
async def get_pr_detail(pr_number: int, repo: str = Query(...)):
    ...

# Option B: Change path to include repo
@router.get("/{repo}/{pr_number}")
async def get_pr_detail(repo: str, pr_number: int):
    ...
```

**Decision: Option A** (query param) — simpler, backward compatible. If `GITHUB_REPO` is set (legacy mode), the param is optional and defaults to the configured repo.

### 17.10 Rate Limit Considerations

| API | Limit | Usage per refresh |
|-----|-------|-------------------|
| Search API | 30 req/min | 1 request (all team PRs in one query) |
| REST API | 5,000 req/hr | ~N×4 (reviews, comments, files, threads per PR) |
| GraphQL API | 5,000 points/hr | N×1 (threads per PR) |

With ~15 open PRs across repos, each refresh costs: 1 search + 15 full PR fetches + 15×3 enrichment = ~61 REST calls. At 2-min refresh intervals, that's ~1,830/hr — well within limits.

### 17.11 Health Endpoint Update

```python
# Include search API rate limit info
{
  "status": "healthy",
  "github_api": {
    "rate_limit": 5000,
    "remaining": 4800,
    "reset_at": 1780832635
  },
  "search_api": {
    "rate_limit": 30,
    "remaining": 28,
    "reset_at": 1780832700
  },
  "mode": "multi-repo"  # or "single-repo" if GITHUB_REPO is set
}